In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

: 

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/train.csv')

print(df.shape)
print(df.head())
print(df.dtypes)


In [ ]:
print(df.isnull().sum())
print(df['Status'].value_counts())
print(df['Edema'].unique())

In [ ]:
print(df.groupby('Status')['Cholesterol'].apply(lambda x: x.isnull().mean()))

In [ ]:
#df['Cholesterol'].fillna(df.groupby('Stage')['Cholesterol'].transform('median'))
df = df[df['Status'] != 'Y']
print(df['Status'].value_counts())


In [ ]:
df['Age']=df['Age']  / 365
print(df['Age'].describe())

In [ ]:
df[(df['Age'] <= 18) | (df['Age'] >=100)]

In [ ]:
print(len(df[(df['Age'] < 18) | (df['Age'] > 100)]))

## Baseline pipeline (CV + submission)
Reload train/test, build a simple reproducible pipeline, evaluate log loss and write `submission.csv`.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
N_SPLITS = 5

In [ ]:
# Kaggle notebook path
TRAIN_PATH = '/kaggle/input/competitions/multiclassificationtask/train.csv'
TEST_PATH = '/kaggle/input/competitions/multiclassificationtask/test.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Target distribution (raw):')
print(train_df['Status'].value_counts(dropna=False))

In [ ]:
# Competition needs 3 classes only: C, CL, D
train_df = train_df[train_df['Status'].isin(['C', 'CL', 'D'])].copy()

X = train_df.drop(columns=['Status'])
y = train_df['Status']
X_test = test_df.copy()

# Optional feature engineering copied from exploration
X['Age_years'] = X['Age'] / 365.0
X_test['Age_years'] = X_test['Age'] / 365.0

# Keep original id for submission
test_ids = X_test['id'].copy()

# We don't want id to influence the model directly
X = X.drop(columns=['id'])
X_test = X_test.drop(columns=['id'])

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = [c for c in X.columns if c not in categorical_cols]

print('Categorical columns:', len(categorical_cols))
print('Numerical columns:', len(numerical_cols))

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=RANDOM_STATE)),
    ]
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_pred = np.zeros((len(X), y.nunique()))
label_order = sorted(y.unique())

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model.fit(X_train, y_train)
    pred_valid = model.predict_proba(X_valid)

    # Align columns by classifier classes_
    class_to_idx = {c: i for i, c in enumerate(model.named_steps['clf'].classes_)}
    aligned = np.column_stack([pred_valid[:, class_to_idx[c]] for c in label_order])
    oof_pred[valid_idx, :] = aligned

    fold_loss = log_loss(y_valid, aligned, labels=label_order)
    print(f'Fold {fold} log loss: {fold_loss:.6f}')

overall_loss = log_loss(y, oof_pred, labels=label_order)
print(f'\nOOF log loss: {overall_loss:.6f}')
print('Label order used:', label_order)

In [ ]:
# Train on full data and create submission
model.fit(X, y)
test_pred = model.predict_proba(X_test)

class_to_idx_full = {c: i for i, c in enumerate(model.named_steps['clf'].classes_)}

submission = pd.DataFrame({
    'id': test_ids,
    'Status_C': test_pred[:, class_to_idx_full['C']],
    'Status_CL': test_pred[:, class_to_idx_full['CL']],
    'Status_D': test_pred[:, class_to_idx_full['D']],
})

# Safety normalization (row-wise) for stable probabilities
prob_cols = ['Status_C', 'Status_CL', 'Status_D']
submission[prob_cols] = submission[prob_cols].div(submission[prob_cols].sum(axis=1), axis=0)

submission.to_csv('/kaggle/working/submission.csv', index=False)
submission.head()